In [7]:
import os
import nest_asyncio
from dotenv import load_dotenv
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision

# Keep Cohere for embeddings
from langchain_cohere import CohereEmbeddings

# Import the new Judge LLM
from langchain_google_genai import ChatGoogleGenerativeAI
# from langchain_openai import ChatOpenAI  <-- Use this if you chose OpenAI

# Allow async execution
nest_asyncio.apply()

# Load API keys
load_dotenv(dotenv_path="../.env")
cohere_api_key = os.getenv("COHERE_API_KEY")

# 1. Initialize Embeddings (Keep Cohere)
evaluator_embeddings = CohereEmbeddings(model="embed-multilingual-v3.0", cohere_api_key=cohere_api_key)

# 2. Initialize the Judge LLM (Swapped to Gemini!)
google_api_key = os.getenv("GOOGLE_API_KEY")
evaluator_llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", google_api_key=google_api_key, temperature=0.0)

# IF USING OPENAI INSTEAD, USE THIS:
# openai_api_key = os.getenv("OPENAI_API_KEY")
# evaluator_llm = ChatOpenAI(model="gpt-4o-mini", api_key=openai_api_key, temperature=0.0)

C:\Users\mansi\AppData\Local\Temp\ipykernel_29852\754527419.py:6: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, answer_relevancy, context_precision
C:\Users\mansi\AppData\Local\Temp\ipykernel_29852\754527419.py:6: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import faithfulness, answer_relevancy, context_precision
C:\Users\mansi\AppData\Local\Temp\ipykernel_29852\754527419.py:6: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import c

In [ ]:
# Replace your current data dictionary with this one
data = {
    "question": [
        # Test 1: Highly specific numeric lookup (Tests Hybrid/BM25 Search)
        "What exact values were used for the Adam optimizer's beta1 and beta2 parameters?",
        
        # Test 2: Conceptual reasoning (Tests Dense Search & Reranking)
        "Why did the authors choose to use Multi-Head Attention instead of a single attention function?",
        
        # Test 3: System architecture detail (Tests Query Expansion)
        "What is the dimensionality of the input and output layers in the base model?"
    ],
    "answer": [
        # These are simulated "perfect" bot outputs. 
        # In reality, you would copy-paste exactly what your Streamlit app generates for these questions.
        "The exact values used for the Adam optimizer's beta1 and beta2 parameters were 0.9 and 0.98, respectively.",
        "The authors chose to use Multi-Head Attention instead of a single attention function because it allows the model to jointly attend to information from different representation subspaces at different positions. With a single attention head, averaging inhibits this ability. By using multiple attention heads, the model can capture diverse and complementary information, leading to improved performance.",
        "The dimensionality of the input and output layers in the base model is dmodel = 512."
    ],
    "contexts": [
        # These are the perfect "gold standard" chunks your retriever SHOULD find.
        ["bottom line of table 3), step time was 1.0 seconds. The big models were trained for 300,000 steps (3.5 days). 5.3 Optimizer We used the Adam optimizer [20] with β1 = 0.9, β2 = 0.98 and ϵ = 10−9. We varied the learning rate over the course of training, according to the formula: lrate = d−0.5 model · min(step_num−0.5, step_num · warmup_steps−1.5) (3) This corresponds to increasing the learning rate linearly for the first warmup_steps training steps, and decreasing it thereafter proportionally to the inverse square root of the step number. We used warmup_steps = 4000. 5.4 Regularization We employ three types of regularization during training: 7 Table 2: The Transformer achieves better BLEU scores than previous state-of-the-art models on the English-to-German and English-to-French newstest2014 tests at a fraction of the training cost. Model BLEU Training Cost (FLOPs) EN-DE EN-FR EN-DE EN-FR ByteNet [18] 23.75 Deep-Att + PosUnk [39] 39.2 1.0 · 1020 GNMT + RL [38] 24.6 39.92 2.3 · 1019", "the limits of language modeling. arXiv preprint arXiv:1602.02410, 2016. [16] Łukasz Kaiser and Samy Bengio. Can active memory replace attention? In Advances in Neural Information Processing Systems, (NIPS), 2016. [17] Łukasz Kaiser and Ilya Sutskever. Neural GPUs learn algorithms. In International Conference on Learning Representations (ICLR), 2016. [18] Nal Kalchbrenner, Lasse Espeholt, Karen Simonyan, Aaron van den Oord, Alex Graves, and Ko- ray Kavukcuoglu. Neural machine translation in linear time. arXiv preprint arXiv:1610.10099v2,[19] Yoon Kim, Carl Denton, Luong Hoang, and Alexander M. Rush. Structured attention networks. In International Conference on Learning Representations, 2017. [20] Diederik Kingma and Jimmy Ba. Adam: A method for stochastic optimization. In ICLR, 2015. [21] Oleksii Kuchaiev and Boris Ginsburg. Factorization tricks for LSTM networks. arXiv preprint arXiv:1703.10722, 2017.", "To evaluate the importance of different components of the Transformer, we varied our base model in different ways, measuring the change in performance on English-to-German translation on the 5We used values of 2.8, 3.7, 6.0 and 9.5 TFLOPS for K80, K40, M40 and P100, respectively. 8 Table 3: Variations on the Transformer architecture. Unlisted values are identical to those of the base model. All metrics are on the English-to-German translation development set, newstest2013. Listed perplexities are per-wordpiece, according to our byte-pair encoding, and should not be compared to per-word perplexities. N dmodel dff h dk dv Pdrop ϵls train PPL BLEU params steps (dev) (dev) ×106 base 6 512 2048 8 64 64 0.1 0.1 100K 4.92 25.8 65 (A) 1 512 512 5.29 24.9 4 128 128 5.00 25.5 16 32 32 4.91 25.8 32 16 16 5.01 25.4 (B) 16 5.16 25.1 58 32 5.01 25.4 60 (C) 2 6.11 23.7 36 4 5.19 25.3 50 8 4.88 25.5 80 256 32 32 5.75 24.5 28 1024 128 128 4.66 26.0 168 1024 5.12 25.4 53 4096 4.75 26.2 90 (D) 0.0 5.77"],
        ["of 1 √dk . Additive attention computes the compatibility function using a feed-forward network with a single hidden layer. While the two are similar in theoretical complexity, dot-product attention is much faster and more space-efficient in practice, since it can be implemented using highly optimized matrix multiplication code. While for small values of dk the two mechanisms perform similarly, additive attention outperforms dot product attention without scaling for larger values of dk [3]. We suspect that for large values of dk, the dot products grow large in magnitude, pushing the softmax function into regions where it has extremely small gradients 4. To counteract this effect, we scale the dot products by 1 √dk . 3.2.2 Multi-Head Attention Instead of performing a single attention function with dmodel-dimensional keys, values and queries, we found it beneficial to linearly project the queries, keys and values h times with different, learned", "linear projections to dk, dk and dv dimensions, respectively. On each of these projected versions of queries, keys and values we then perform the attention function in parallel, yielding dv-dimensional 4To illustrate why the dot products get large, assume that the components of q and k are independent random variables with mean 0 and variance 1. Then their dot product, q · k = Pdk i=1 qiki, has mean 0 and variance dk. 4 output values. These are concatenated and once again projected, resulting in the final values, as depicted in Figure 2. Multi-head attention allows the model to jointly attend to information from different representation subspaces at different positions. With a single attention head, averaging inhibits this. MultiHead(Q, K, V ) = Concat(head1, ..., headh)W O where headi = Attention(QW Q i , KW K i , V W V i ) Where the projections are parameter matrices W Q i ∈Rdmodel×dk, W K i ∈Rdmodel×dk, W V i ∈Rdmodel×dv and W O ∈Rhdv×dmodel.", "around each of the sub-layers, followed by layer normalization. We also modify the self-attention sub-layer in the decoder stack to prevent positions from attending to subsequent positions. This masking, combined with fact that the output embeddings are offset by one position, ensures that the predictions for position i can depend only on the known outputs at positions less than i. 3.2 Attention An attention function can be described as mapping a query and a set of key-value pairs to an output, where the query, keys, values, and output are all vectors. The output is computed as a weighted sum 3 Scaled Dot-Product Attention Multi-Head Attention Figure 2: (left) Scaled Dot-Product Attention. (right) Multi-Head Attention consists of several attention layers running in parallel. of the values, where the weight assigned to each value is computed by a compatibility function of the query with the corresponding key. 3.2.1 Scaled Dot-Product Attention"],
        ["FFN(x) = max(0, xW1 + b1)W2 + b2 (2) While the linear transformations are the same across different positions, they use different parameters from layer to layer. Another way of describing this is as two convolutions with kernel size 1. The dimensionality of input and output is dmodel = 512, and the inner-layer has dimensionality dff = 2048. 3.4 Embeddings and Softmax Similarly to other sequence transduction models, we use learned embeddings to convert the input tokens and output tokens to vectors of dimension dmodel. We also use the usual learned linear transfor- mation and softmax function to convert the decoder output to predicted next-token probabilities. In our model, we share the same weight matrix between the two embedding layers and the pre-softmax linear transformation, similar to [30]. In the embedding layers, we multiply those weights by √dmodel. 5 Table 1: Maximum path lengths, per-layer complexity and minimum number of sequential operations", "6 length n is smaller than the representation dimensionality d, which is most often the case with sentence representations used by state-of-the-art models in machine translations, such as word-piece [38] and byte-pair [31] representations. To improve computational performance for tasks involving very long sequences, self-attention could be restricted to considering only a neighborhood of size r in the input sequence centered around the respective output position. This would increase the maximum path length to O(n/r). We plan to investigate this approach further in future work. A single convolutional layer with kernel width k < n does not connect all pairs of input and output positions. Doing so requires a stack of O(n/k) convolutional layers in the case of contiguous kernels, or O(logk(n)) in the case of dilated convolutions [18], increasing the length of the longest paths between any two positions in the network. Convolutional layers are generally more expensive than", "i ∈Rdmodel×dk, W K i ∈Rdmodel×dk, W V i ∈Rdmodel×dv and W O ∈Rhdv×dmodel. In this work we employ h = 8 parallel attention layers, or heads. For each of these we use dk = dv = dmodel/h = 64. Due to the reduced dimension of each head, the total computational cost is similar to that of single-head attention with full dimensionality. 3.2.3 Applications of Attention in our Model The Transformer uses multi-head attention in three different ways: • In 'encoder-decoder attention' layers, the queries come from the previous decoder layer, and the memory keys and values come from the output of the encoder. This allows every position in the decoder to attend over all positions in the input sequence. This mimics the typical encoder-decoder attention mechanisms in sequence-to-sequence models such as [38, 2, 9]. • The encoder contains self-attention layers. In a self-attention layer all of the keys, values and queries come from the same place, in this case, the output of the previous layer in the"]
    ],
    "ground_truth": [
        # The absolute factual truth from the paper that the evaluator LLM uses to grade the response
        "beta1 = 0.9 and beta2 = 0.98",
        "It allows the model to jointly attend to information from different representation subspaces at different positions.",
        "The dimensionality (d_model) is 512."
    ]
}

# Convert it into a HuggingFace Dataset object
eval_dataset = Dataset.from_dict(data)
print("Transformer benchmark dataset created successfully!")

Transformer benchmark dataset created successfully!


In [11]:
from ragas.run_config import RunConfig

# FIX 2: Throttling the requests to respect the free-tier rate limits
results = evaluate(
    dataset=eval_dataset,
    metrics=[
        context_precision,
        faithfulness,
        answer_relevancy,
    ],
    llm=evaluator_llm,
    embeddings=evaluator_embeddings,
    run_config=RunConfig(max_workers=1, max_retries=5) # Forces single-threaded execution
)

# Display the final scores
print("--- Final Evaluation Scores ---")
df_results = results.to_pandas()
display(df_results)

Evaluating: 100%|██████████| 9/9 [10:50<00:00, 72.24s/it] 


--- Final Evaluation Scores ---


,user_input,retrieved_contexts,response,reference,context_precision,faithfulness,answer_relevancy
0,What exact values were used for the Adam optim...,"[bottom line of table 3), step time was 1.0 se...",The exact values used for the Adam optimizer's...,beta1 = 0.9 and beta2 = 0.98,1.0,1.0,0.880216
1,Why did the authors choose to use Multi-Head A...,[of 1 √dk . Additive attention computes the co...,The authors chose to use Multi-Head Attention ...,It allows the model to jointly attend to infor...,0.5,0.8,0.878942
2,What is the dimensionality of the input and ou...,"[FFN(x) = max(0, xW1 + b1)W2 + b2 (2) While th...",The dimensionality of the input and output lay...,The dimensionality (d_model) is 512.,NaN,NaN,NaN


In [20]:
import sys
import os
import nest_asyncio
import pandas as pd
from dotenv import load_dotenv
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness, context_precision
from ragas.run_config import RunConfig

from langchain_cohere import CohereEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI

# Allow async execution in Jupyter
nest_asyncio.apply()

# Add the project root to the system path so we can import from src
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

from src.vectorstore import VectorStore
from src.chatbot import Chatbot

C:\Users\mansi\AppData\Local\Temp\ipykernel_29852\3888004808.py:8: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, context_precision
C:\Users\mansi\AppData\Local\Temp\ipykernel_29852\3888004808.py:8: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import faithfulness, context_precision


In [22]:
# Load API keys from your .env
load_dotenv(dotenv_path="../.env")
cohere_api_key = os.getenv("COHERE_API_KEY")
google_api_key = os.getenv("GOOGLE_API_KEY")
pinecone_api_key = os.getenv("PINECONE_API_KEY")

# Initialize your custom pipeline
pdf_path = "C:/Users/mansi/Downloads/attention is all you need.pdf"

print("Initializing VectorStore (This may take a moment if it's embedding the PDF)...")
vectorstore = VectorStore(
    pdf_path=pdf_path,
    cohere_api_key=cohere_api_key,
    pinecone_api_key=pinecone_api_key,
    namespace="eval-session"
)

print("Initializing Chatbot...")
chatbot = Chatbot(vectorstore, cohere_api_key)
print("Pipeline Ready!")

Initializing VectorStore (This may take a moment if it's embedding the PDF)...


100%|██████████| 44/44 [00:00<00:00, 206.62it/s]


Initializing Chatbot...
Pipeline Ready!


In [23]:
# Initialize Embeddings
evaluator_embeddings = CohereEmbeddings(
    model="embed-multilingual-v3.0", 
    cohere_api_key=cohere_api_key
)

# Initialize Gemini as the Judge (with high timeout for rate limits)
evaluator_llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash", 
    google_api_key=google_api_key, 
    temperature=0.0,
    timeout=120,
    max_retries=6
)

In [28]:
# --- MICRO-BENCHMARK ---
# We use the hardest numeric question to prove the BM25 Hybrid Search delta

questions = [
    "What exact values were used for the Adam optimizer's beta1 and beta2 parameters?"
]

ground_truths = [
    "beta1 = 0.9 and beta2 = 0.98"
]

In [29]:
# --- LOOP 1: NAIVE RAG (BASELINE) ---
baseline_answers = []
baseline_contexts = []

print("Running Baseline Evaluation...")
for q in questions:
    contexts = vectorstore.retrieve_baseline(q) 
    
    # Generate response using standard chat (no agentic logic)
    res = chatbot.co.chat(
        message=q, 
        documents=[{"text": c['text']} for c in contexts], 
        model="command-a-03-2025",
        temperature=0.0
    )
    
    baseline_answers.append(res.text)
    baseline_contexts.append([c['text'] for c in contexts])

baseline_dataset = Dataset.from_dict({
    "question": questions,
    "answer": baseline_answers,
    "contexts": baseline_contexts,
    "ground_truth": ground_truths
})
print("Baseline Data Collected!")

Running Baseline Evaluation...
Baseline Data Collected!


In [30]:
# --- LOOP 2: ADVANCED RAG (OPTIMIZED) ---
optimized_answers = []
optimized_contexts = []

print("Running Optimized Evaluation...")
for q in questions:
    response_stream, contexts = chatbot.respond(q, chat_history=[])
    
    full_text = ""
    for event in response_stream:
        if event.event_type == "text-generation":
            full_text += event.text
            
    optimized_answers.append(full_text)
    optimized_contexts.append([c['text'] for c in contexts])

optimized_dataset = Dataset.from_dict({
    "question": questions,
    "answer": optimized_answers,
    "contexts": optimized_contexts,
    "ground_truth": ground_truths
})
print("Optimized Data Collected!")

Running Optimized Evaluation...
[Router raw output]: 'RAG'
[Router decision]: RAG pipeline
Expanded Queries: ["What exact values were used for the Adam optimizer's beta1 and beta2 parameters?", 'What were the specific beta1 and beta2 hyperparameters set to in the Adam optimization algorithm?', 'Can you provide the numerical values assigned to the beta1 and beta2 coefficients in the Adam optimizer?', "What are the default or recommended values for the Adam optimizer's beta1 and beta2 exponential decay rates?"]
Total unique chunks retrieved before reranking: 9
Optimized Data Collected!


In [31]:
# Throttled config for free-tier API
config = RunConfig(max_workers=1, max_retries=5)

print("Calculating Baseline Scores (this takes a minute)...")
baseline_results = evaluate(
    dataset=baseline_dataset,
    metrics=[context_precision, faithfulness],
    llm=evaluator_llm,
    embeddings=evaluator_embeddings,
    run_config=config
).to_pandas()

print("Calculating Optimized Scores (this takes a minute)...")
optimized_results = evaluate(
    dataset=optimized_dataset,
    metrics=[context_precision, faithfulness],
    llm=evaluator_llm,
    embeddings=evaluator_embeddings,
    run_config=config
).to_pandas()

# --- FINAL OUTPUT ---
print("\n" + "="*40)
print("FINAL COMPARISON SUMMARY")
print("="*40)

summary = pd.DataFrame({
    "Metric": ["Context Precision", "Faithfulness"],
    "Baseline (Naive RAG)": [baseline_results['context_precision'].mean(), baseline_results['faithfulness'].mean()],
    "Optimized (Hybrid + Multi-Query)": [optimized_results['context_precision'].mean(), optimized_results['faithfulness'].mean()]
})

summary["Improvement %"] = ((summary["Optimized (Hybrid + Multi-Query)"] - summary["Baseline (Naive RAG)"]) / summary["Baseline (Naive RAG)"]) * 100
display(summary)

Calculating Baseline Scores (this takes a minute)...


Evaluating: 100%|██████████| 2/2 [03:00<00:00, 90.01s/it]


Calculating Optimized Scores (this takes a minute)...


Evaluating: 100%|██████████| 2/2 [06:00<00:00, 180.00s/it]



FINAL COMPARISON SUMMARY


,Metric,Baseline (Naive RAG),Optimized (Hybrid + Multi-Query),Improvement %
0,Context Precision,0.0,NaN,NaN
1,Faithfulness,NaN,NaN,NaN
